In [5]:

!pip install chemicals

  Using cached fluids-1.3.0-py3-none-any.whl (608 kB)


In [8]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.integrate import quad
import chemicals

# Universal Gas Constant
R = 8.31446261815324  # J / (mol K)

def get_eos_params(eos_type, Tc, Pc, omega, T):
    """
    Calculates the generalized cubic EOS parameters for a given substance.
    Supports Peng-Robinson (PR) and Soave-Redlich-Kwong (SRK).
    """
    if eos_type == 'PR':
        u, w = 2, -1
        Omega_a = 0.457235
        Omega_b = 0.077796
        kappa = 0.37464 + 1.54226 * omega - 0.26992 * omega**2
    elif eos_type == 'SRK':
        u, w = 1, 0
        Omega_a = 0.42748
        Omega_b = 0.08664
        kappa = 0.480 + 1.574 * omega - 0.176 * omega**2
    else:
        raise ValueError("Unsupported EOS type. Choose 'PR' or 'SRK'.")

    # Alpha function (temperature dependence of attractive parameter 'a')
    alpha = (1 + kappa * (1 - np.sqrt(T / Tc)))**2
    
    a = Omega_a * (R * Tc)**2 / Pc * alpha
    b = Omega_b * R * Tc / Pc
    
    return u, w, a, b

def pressure_eos(v, T, a, b, u, w):
    """Generic Cubic Equation of State."""
    return (R * T) / (v - b) - a / (v**2 + u * b * v + w * b**2)

def solve_z_cubic(A, B_z, u, w):
    """Solves the cubic polynomial for the compressibility factor Z (Eqn 6)."""
    # Polynomial coefficients for: Z^3 + C2*Z^2 + C1*Z + C0 = 0
    C2 = (u - 1) * B_z - 1
    C1 = A - u * B_z**2 - u * B_z + w * B_z**2
    C0 = -(A * B_z + w * B_z**2 + w * B_z**3)
    
    # Find roots
    roots = np.roots([1, C2, C1, C0])
    
    # Filter for real, positive roots
    real_roots = np.real(roots[np.iscomplex(roots) == False])
    real_roots = real_roots[real_roots > 0]
    
    return real_roots

def calculate_saturation_state(T, Tc, Pc, omega, eos_type='PR', tol=1e-6, max_iter=200):
    """
    Implements the iterative algorithm from the flowchart to find Ps, vl, and vv.
    """
    u, w, a, b = get_eos_params(eos_type, Tc, Pc, omega, T)
    
    # 1. Asumir presión de saturación Ps (Initial guess using Wilson's equation)
    Ps = Pc * np.exp(5.373 * (1 + omega) * (1 - Tc / T))
    
    for iteration in range(max_iter):
        # Dimensionless parameters
        A = (a * Ps) / (R * T)**2
        B_z = (b * Ps) / (R * T)
        
        # 2. Encontrar raíces de la ecn. 6
        z_roots = solve_z_cubic(A, B_z, u, w)
        
        # If less than 3 real roots, we are outside the two-phase region
        if len(z_roots) < 3:
            return None, None, None
            
        z_min = np.min(z_roots)
        z_max = np.max(z_roots)
        
        # 3. Calcular volumen del líquido y vapor ecn. 7
        vl = z_min * (R * T) / Ps
        vv = z_max * (R * T) / Ps
        
        # 4. Resolver la integral de la ecn. 5
        # Using SciPy's numerical integration for exact precision to avoid manual analytical typos
        integral_val, _ = quad(pressure_eos, vl, vv, args=(T, a, b, u, w))
        
        # 5. Calcular nueva presión de saturación ecn. 4
        Ps_new = integral_val / (vv - vl)
        
        # 6. Check convergence (es tau_1 < epsilon_1 ?)
        error = abs(Ps_new - Ps) / Ps
        if error < tol:
            return Ps_new, vl, vv
            
        # 7. Con última Ps calculada (Update and loop)
        # Using a slight relaxation factor to ensure convergence stability
        Ps = 0.5 * Ps_new + 0.5 * Ps
        
    return None, None, None

def generate_equilibrium_data(substance_name, Tc, Pc, omega, eos_type='PR', T_min_ratio=0.55, num_points=40):
    """Generates the equilibrium curve table for a range of temperatures."""
    temperatures = np.linspace(Tc * T_min_ratio, Tc * 0.995, num_points)
    
    data = []
    for T in temperatures:
        Ps, vl, vv = calculate_saturation_state(T, Tc, Pc, omega, eos_type)
        if Ps is not None:
            data.append({
                'T (K)': round(T, 2),
                'P_sat (Pa)': Ps,
                'P_sat (MPa)': round(Ps / 1e6, 4),
                'v_l (m3/mol)': vl,
                'v_v (m3/mol)': vv
            })
            
    df = pd.DataFrame(data)
    return df

def plot_pv_diagram(df, substance_name, Tc, Pc, omega, eos_type='PR'):
    """Generates an interactive P-v diagram using Plotly."""
    fig = go.Figure()

    # Saturated Liquid Line
    fig.add_trace(go.Scatter(
        x=df['v_l (m3/mol)'], y=df['P_sat (MPa)'],
        mode='lines+markers', name='Saturated Liquid',
        line=dict(color='blue', width=3),
        marker=dict(size=4)
    ))

    # Saturated Vapor Line
    fig.add_trace(go.Scatter(
        x=df['v_v (m3/mol)'], y=df['P_sat (MPa)'],
        mode='lines+markers', name='Saturated Vapor',
        line=dict(color='red', width=3),
        marker=dict(size=4)
    ))

    # Calculate and plot a few Isotherms
    v_range = np.logspace(np.log10(df['v_l (m3/mol)'].min()*0.9), np.log10(df['v_v (m3/mol)'].max()*1.5), 200)
    
    # Plot Isotherms (Subcooled, Critical, Supercritical)
    u, w, _, _ = get_eos_params(eos_type, Tc, Pc, omega, Tc) # Just to get u, w
    T_isotherms = [df['T (K)'].iloc[0], df['T (K)'].iloc[len(df)//2], Tc, Tc*1.1]
    colors = ['rgba(150,150,150,0.5)', 'rgba(150,150,150,0.5)', 'black', 'rgba(255,165,0,0.8)']
    names = [f"T = {T_isotherms[0]} K", f"T = {T_isotherms[1]} K", "Critical Isotherm", f"T = {T_isotherms[3]:.1f} K"]
    
    for T_iso, color, name in zip(T_isotherms, colors, names):
        _, _, a, b = get_eos_params(eos_type, Tc, Pc, omega, T_iso)
        P_iso = [pressure_eos(v, T_iso, a, b, u, w) / 1e6 for v in v_range]
        fig.add_trace(go.Scatter(
            x=v_range, y=P_iso,
            mode='lines', name=name,
            line=dict(color=color, dash='dash')
        ))

    # Layout configurations
    fig.update_layout(
        title=f"P-v Diagram for {substance_name} (Using {eos_type} EOS)",
        xaxis_title="Molar Volume, v (m³/mol)",
        yaxis_title="Pressure, P (MPa)",
        xaxis_type="log",  # Log scale is standard for P-v diagrams due to large vapor volumes
        yaxis=dict(range=[0, df['P_sat (MPa)'].max() * 1.5]),
        template="plotly_white",
        legend=dict(x=0.75, y=0.95)
    )
    
    fig.show()

# ==========================================
# EXAMPLE EXECUTION: Generic Parameters
# Let's use Water (H2O) as the test substance
# ==========================================
if __name__ == "__main__":
    # Substance parameters (Water)
    substance = "n-Heptane"
    CAS_Number = chemicals.search_chemical(substance).CASs
    T_crit = chemicals.Tc(CAS_Number)      # K
    P_crit = chemicals.Pc(CAS_Number)     # Pa
    acentric_factor = chemicals.omega(CAS_Number)
    
    # 1. Generate Table of Equilibrium Data
    print(f"Calculating Equilibrium Data for {substance}...")
    df_equilibrium = generate_equilibrium_data(
        substance_name=substance, 
        Tc=T_crit, 
        Pc=P_crit, 
        omega=acentric_factor, 
        eos_type='PR' # Peng-Robinson
    )
    
    print("\n--- Equilibrium Data Table ---")
    print(df_equilibrium[['T (K)', 'P_sat (MPa)', 'v_l (m3/mol)', 'v_v (m3/mol)']].to_string(index=False))
    
    # 2. Generate Interactive Plot
    plot_pv_diagram(df_equilibrium, substance, T_crit, P_crit, acentric_factor, eos_type='PR')

Calculating Equilibrium Data for n-Heptane...

--- Equilibrium Data Table ---
 T (K)  P_sat (MPa)  v_l (m3/mol)  v_v (m3/mol)
297.11       0.0060      0.000149      0.411418
303.27       0.0080      0.000150      0.311719
309.44       0.0107      0.000151      0.239312
315.60       0.0140      0.000152      0.185989
321.77       0.0181      0.000153      0.146206
327.93       0.0231      0.000154      0.116159
334.09       0.0293      0.000155      0.093205
340.26       0.0367      0.000157      0.075478
346.42       0.0456      0.000158      0.061648
352.58       0.0561      0.000159      0.050756
358.75       0.0685      0.000161      0.042100
364.91       0.0830      0.000162      0.035161
371.08       0.0998      0.000164      0.029556
377.24       0.1191      0.000166      0.024992
383.40       0.1414      0.000167      0.021250
389.57       0.1667      0.000169      0.018161
395.73       0.1954      0.000171      0.015595
401.89       0.2278      0.000173      0.013450
408.06    